---
title: Research of municipality identifiers in Germany
date: now
author: Jan Cap
---

Germany has online portal for getting some basic information about municipalities: https://www.statistikportal.de/de/gemeindeverzeichnis.
On this portal, you can get municipality ID which consists of several parts representing different administrative levels:
- Land (2 digits) - federal state
- Regierungsbezirk (1 digit) - administrative region
- Kreis (2 digits) - district (county)
- Verbandsgemeinde (4 digits) - collective municipality
- Gemeinde (3 digits) - municipality (local community)

Data also contain zip codes, population by sex, area in km², population density, and so on.


According to https://www.destatis.de/EN/Themes/Countries-Regions/Regional-Statistics/OnlineListMunicipalities/_inhalt.html#417214 there is 10754 municipalities in Germany as of december 2024. On the destatis portal, data are provided anually and quarterly. We explored data from december 2024. Data are provided in excel format and they contain all geographical levels. 

## Destatis portal

We explored the data from destatis portal and we found out that there are more records than expected. It is due to the fact that in the excel, there are all geographical levels. Data contain columns for Land, Regierungsbezirk, Kreis, Verbandsgemeinde and Gemeinde. So there are records for each level of administrative division. We try to filter out only records for municipalities (Gemeinde) by filtering out records with empty value in the column "Gem". After filtering, we have 10959 records, which is still lot more than expected 10754 municipalities.

In [ ]:
import pandas as pd

# sheet Onlineprodukt_Gemeinden31122024
df = pd.read_excel(
    "../data/raw/31122024_Auszug_GV.xlsx",
    sheet_name="Onlineprodukt_Gemeinden31122024",
    skiprows=3,
    dtype={"Land": str, "RB": str, "Kreis": str, "Gem": str},
)
df

In [ ]:
# filter out records with empty value in the column "Gem"
df = df[df["Gem"].notna()]
df

In [ ]:
df["AGS"] = df["Land"] + df["RB"] + df["Kreis"] + df["Gem"]
df

## Census data

In [ ]:
from geoscore_de.data_flow.features.municipality import load_municipality_data

df = load_municipality_data("../data/raw/municipalities_2022.csv")

In [ ]:
df.head()

In [ ]:
df.sort_values(by="Persons", ascending=False).head(30)

In [ ]:
num_of_municipalities = df["MU_ID"].nunique()
print(f"Number of unique municipalities: {num_of_municipalities}")

This data can be used as a reference for data on municipalities in Germany.
It contains correct municipality IDs for all municipalities in Germany as of 2022.
Use this dataframe for grouping and joining with other datasets to ensure correct municipality IDs are used.

Lets try to create id without Verbandsgemeinde (collective municipality) level and see if the counts match.

In [ ]:
# AGS is official municipality key
df["AGS"] = df["MU_ID"].str.slice(0, 5) + df["MU_ID"].str.slice(9, 12)
df

In [ ]:
print(f"Number of unique AGS keys: {df['AGS'].nunique()}")

In [ ]:
# number of municipalities per state
df["State"] = df["AGS"].str.slice(0, 2)
states = df["State"].value_counts().sort_index().to_frame()
states["pct"] = states / states.sum() * 100
print(states)

Count of unique AGS keys is the same as count of unique MU_ID keys without Verbandsgemeinde level.